In [1]:
"""
训练CatBoost的三种模型，FitAll，FitOne，Dynamic
"""
import os
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import pearsonr

IN_PATH = "/data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/trainTestData/"

traits = [
    "Yield_per_plant",
    "1000-grain_weight",
    "Grain_length_to_width_ratio",
    "Heading_date",
    "Plant_height",
    "Grain_per_panicle",
    "Seed_set_rate",
    "Tiller_number"
]

In [2]:
results = []

for target_col in traits:

    print("Running trait:", target_col)

    train_df = pd.read_table(os.path.join(IN_PATH, f"train-{target_col}.txt"))
    test_df  = pd.read_table(os.path.join(IN_PATH, f"test-{target_col}.txt"))

    train_df["location"] = train_df["location"].astype(str)
    train_df["year"] = train_df["year"].astype(str)

    test_df["location"] = test_df["location"].astype(str)
    test_df["year"] = test_df["year"].astype(str)

    feature_cols = [c for c in train_df.columns if c.startswith("fea_")]
    cat_features = ["location","year"]

    feature_cols = feature_cols + cat_features

    pairs = (
        train_df[["year","location"]]
        .drop_duplicates()
        .sort_values(["year","location"])
    )

    pairs = list(pairs.itertuples(index=False, name=None))

    # =============================
    # TrainingWhole
    # =============================

    whole_pool = Pool(
        data=train_df[feature_cols],
        label=train_df[target_col],
        cat_features=cat_features
    )

    whole_model = CatBoostRegressor(
        iterations=1000,
        thread_count=48,
        verbose=0
    )

    whole_model.fit(whole_pool)

    # =============================
    # TrainingTwice 第一阶段
    # =============================

    first_model = CatBoostRegressor(
        iterations=1000,
        thread_count=48,
        verbose=0
    )

    first_model.fit(whole_pool)

    # =============================
    # 按环境测试
    # =============================

    for i,(year,location) in enumerate(pairs):

        test_data = test_df[
            (test_df["year"]==year) &
            (test_df["location"]==location)
        ]

        if len(test_data) == 0:
            continue

        test_pool = Pool(
            data=test_data[feature_cols],
            label=test_data[target_col],
            cat_features=cat_features
        )

        y_true = test_data[target_col].values

        # =============================
        # TrainingWhole
        # =============================

        preds = whole_model.predict(test_pool)

        whole_r2 = r2_score(y_true, preds)
        whole_rmse = np.sqrt(mean_squared_error(y_true, preds))
        whole_pcc = pearsonr(y_true, preds)[0]

        # =============================
        # TrainingOnebyOne
        # =============================

        train_data = train_df[
            (train_df["year"]==year) &
            (train_df["location"]==location)
        ]

        train_pool = Pool(
            data=train_data[feature_cols],
            label=train_data[target_col],
            cat_features=cat_features
        )

        one_model = CatBoostRegressor(
            iterations=1000,
            thread_count=48,
            verbose=0
        )

        one_model.fit(train_pool)

        preds = one_model.predict(test_pool)

        one_r2 = r2_score(y_true, preds)
        one_rmse = np.sqrt(mean_squared_error(y_true, preds))
        one_pcc = pearsonr(y_true, preds)[0]

        # =============================
        # TrainingTwice
        # =============================

        sec_model = CatBoostRegressor(
            iterations=1000,
            thread_count=48,
            verbose=0
        )

        sec_model.fit(train_pool, init_model=first_model)

        preds = sec_model.predict(test_pool)

        twice_r2 = r2_score(y_true, preds)
        twice_rmse = np.sqrt(mean_squared_error(y_true, preds))
        twice_pcc = pearsonr(y_true, preds)[0]

        results.append({
            "Trait": target_col,
            "Round": i+1,
            "year": year,
            "location": location,
            "test_data": len(test_data),

            "TraingWhole_R2": whole_r2,
            "TraingWhole_RMSE": whole_rmse,
            "TraingWhole_PCC": whole_pcc,

            "TraingOnebyOne_R2": one_r2,
            "TraingOnebyOne_RMSE": one_rmse,
            "TraingOnebyOne_PCC": one_pcc,

            "TraingTwice_R2": twice_r2,
            "TraingTwice_RMSE": twice_rmse,
            "TraingTwice_PCC": twice_pcc
        })


# =============================
# 输出结果
# =============================

result_df = pd.DataFrame(results)

print(result_df)

#result_df.to_csv("catboost_compare_results.csv", index=False)
result_df.to_csv(os.path.join(IN_PATH, "catboost_compare_results.csv"), index=False)

Running trait: Yield_per_plant
Running trait: 1000-grain_weight
Running trait: Grain_length_to_width_ratio
Running trait: Heading_date
Running trait: Plant_height
Running trait: Grain_per_panicle
Running trait: Seed_set_rate
Running trait: Tiller_number
               Trait  Round    year     location  test_data  TraingWhole_R2  \
0    Yield_per_plant      1  2022.0  Gongzhuling         96        0.389921   
1    Yield_per_plant      2  2022.0      Kunming        274        0.791779   
2    Yield_per_plant      3  2022.0      Nanning        353        0.795891   
3    Yield_per_plant      4  2022.0      Tanghai        290        0.615886   
4    Yield_per_plant      5  2023.0  Gongzhuling        195        0.437699   
..               ...    ...     ...          ...        ...             ...   
99     Tiller_number      9  2023.0        Hefei        360        0.239668   
100    Tiller_number     10  2023.0      Kunming        315        0.389264   
101    Tiller_number     11  2023.0

In [3]:
result_df

,Trait,Round,year,location,test_data,TraingWhole_R2,TraingWhole_RMSE,TraingWhole_PCC,TraingOnebyOne_R2,TraingOnebyOne_RMSE,TraingOnebyOne_PCC,TraingTwice_R2,TraingTwice_RMSE,TraingTwice_PCC
0,Yield_per_plant,1,2022.0,Gongzhuling,96,0.389921,3.793213,0.642582,0.437096,3.643605,0.676731,0.470519,3.533782,0.706489
1,Yield_per_plant,2,2022.0,Kunming,274,0.791779,2.375930,0.891707,0.826774,2.167093,0.909734,0.815269,2.237907,0.903165
2,Yield_per_plant,3,2022.0,Nanning,353,0.795891,3.094089,0.892755,0.788425,3.150170,0.888882,0.798295,3.075812,0.894518
3,Yield_per_plant,4,2022.0,Tanghai,290,0.615886,5.374078,0.786887,0.602517,5.466803,0.777222,0.609054,5.421661,0.780477
4,Yield_per_plant,5,2023.0,Gongzhuling,195,0.437699,3.713593,0.663011,0.450406,3.671392,0.671378,0.482107,3.563938,0.696565
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,Tiller_number,9,2023.0,Hefei,360,0.239668,2.280910,0.495052,0.283182,2.214680,0.534172,0.285785,2.210655,0.535045
100,Tiller_number,10,2023.0,Kunming,315,0.389264,1.574323,0.634388,0.381811,1.583900,0.624936,0.401440,1.558552,0.643677
101,Tiller_number,11,2023.0,Nanning,360,0.568201,0.978223,0.757501,0.572502,0.973339,0.757693,0.605780,0.934688,0.779025
102,Tiller_number,12,2023.0,Tanghai,354,0.379985,1.642489,0.625004,0.361707,1.666524,0.602719,0.437751,1.564105,0.663283
